# Section 2: SKU Aliases & BOM Depth (Q11–18)

SKU supersession chains and BOM hierarchy analysis.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))
from pcg_example.benchmark.notebook_helpers import get_session, run_sql, explode_bom, display_bom_tree
conn, ontology = get_session()

## Q11

SKU code SKU-ORAL-020-OLD was retired. What active SKU replaced it? Show the old and new codes side by side with their names and active status.

In [ ]:
# IDENTIFY: SKU -> SKUSupersedes | DISPATCH: direct_join (depth-1 chain)
# The OLD SKU's supersedes_sku_id points to the NEW replacement
run_sql(conn, """
    SELECT old.sku_code as old_code, old.name as old_name, old.is_active as old_active,
           new.sku_code as new_code, new.name as new_name, new.is_active as new_active
    FROM skus old
    JOIN skus new ON old.supersedes_sku_id = new.id
    WHERE old.sku_code = 'SKU-ORAL-020-OLD'
""")

## Q12

Which finished SKUs have a BOM tree deeper than 2 levels? Show each SKU, the intermediates at each level, and the premix sub-intermediates.

In [ ]:
# IDENTIFY: SKU -> Formula (bom_level=0) -> FormulaHasIngredients -> BulkIntermediate
#           -> Formula (bom_level=1) -> FormulaHasIngredients -> BulkIntermediate (premix, bom_level=2)
# DISPATCH: multi-level SQL joins to find 3-level BOMs
run_sql(conn, """
    SELECT DISTINCT
        s.sku_code,
        s.name as sku_name,
        bi1.bulk_code as level1_intermediate,
        bi1.name as level1_name,
        bi2.bulk_code as level2_premix,
        bi2.name as level2_name
    FROM skus s
    JOIN formulas f0 ON f0.product_id = s.id AND f0.bom_level = 0
    JOIN formula_ingredients fi0 ON fi0.formula_id = f0.id
    JOIN bulk_intermediates bi1 ON fi0.ingredient_id = bi1.id AND bi1.bom_level = 1
    JOIN formulas f1 ON f1.product_id = bi1.id AND f1.bom_level = 1
    JOIN formula_ingredients fi1 ON fi1.formula_id = f1.id
    JOIN bulk_intermediates bi2 ON fi1.ingredient_id = bi2.id AND bi2.bom_level = 2
    ORDER BY s.sku_code, bi1.bulk_code, bi2.bulk_code
""")

## Q13

Which raw materials are diamond dependencies — appearing in both a premix formula AND directly in its parent bulk formula? Show the ingredient name, the premix code, and the parent bulk code.

In [ ]:
# Diamond: ingredient appears in both a bom_level=1 formula directly
# AND in a premix (bom_level=2) that the same formula uses
run_sql(conn, """
    WITH bulk_direct AS (
        SELECT f1.product_id as bulk_id, fi1.ingredient_id, f1.formula_code
        FROM formulas f1
        JOIN formula_ingredients fi1 ON fi1.formula_id = f1.id
        JOIN ingredients i ON fi1.ingredient_id = i.id
        WHERE f1.bom_level = 1
    ),
    premix_ingredients AS (
        SELECT f1.product_id as bulk_id, bi2.id as premix_id, bi2.bulk_code as premix_code,
               fi2.ingredient_id, f1.formula_code as bulk_formula
        FROM formulas f1
        JOIN formula_ingredients fi1 ON fi1.formula_id = f1.id
        JOIN bulk_intermediates bi2 ON fi1.ingredient_id = bi2.id AND bi2.bom_level = 2
        JOIN formulas f2 ON f2.product_id = bi2.id AND f2.bom_level = 2
        JOIN formula_ingredients fi2 ON fi2.formula_id = f2.id
        JOIN ingredients i ON fi2.ingredient_id = i.id
        WHERE f1.bom_level = 1
    )
    SELECT i.ingredient_code, i.name as ingredient_name,
           pi.premix_code,
           bi.bulk_code as parent_bulk_code
    FROM premix_ingredients pi
    JOIN bulk_direct bd ON bd.bulk_id = pi.bulk_id AND bd.ingredient_id = pi.ingredient_id
    JOIN ingredients i ON pi.ingredient_id = i.id
    JOIN bulk_intermediates bi ON pi.bulk_id = bi.id
    ORDER BY bi.bulk_code, pi.premix_code, i.ingredient_code
""")

## Q14

Which SKUs are original root codes — meaning they were never created as a replacement for an older SKU? Give me a count by category.

In [ ]:
# Root SKUs: supersedes_sku_id IS NULL means this SKU was NOT created as a replacement
run_sql(conn, """
    SELECT category, COUNT(*) as root_sku_count
    FROM skus
    WHERE supersedes_sku_id IS NULL
    GROUP BY category
    ORDER BY root_sku_count DESC
""")

## Q15

Which discontinued SKUs have supersedes_sku_id links? How many total alias pairs exist in the system?

In [ ]:
df = run_sql(conn, """
    SELECT sku_code, name, is_active, supersedes_sku_id
    FROM skus
    WHERE supersedes_sku_id IS NOT NULL
    ORDER BY sku_code
""")
display(df)
print(f"\nTotal alias pairs: {len(df)}")

## Q16

Which finished SKUs use more than one bulk intermediate in their formula? Show the SKU, the primary and secondary intermediates, and their weight fractions.

In [ ]:
run_sql(conn, """
    WITH sku_bulks AS (
        SELECT s.sku_code, s.name as sku_name,
               bi.bulk_code, bi.name as bulk_name,
               fi.quantity_kg,
               SUM(fi.quantity_kg) OVER (PARTITION BY s.id) as total_qty
        FROM skus s
        JOIN formulas f ON f.product_id = s.id AND f.bom_level = 0
        JOIN formula_ingredients fi ON fi.formula_id = f.id
        JOIN bulk_intermediates bi ON fi.ingredient_id = bi.id
    )
    SELECT sku_code, sku_name, bulk_code, bulk_name,
           quantity_kg,
           ROUND(quantity_kg / NULLIF(total_qty, 0), 4) as weight_fraction
    FROM sku_bulks
    WHERE sku_code IN (
        SELECT sku_code FROM sku_bulks GROUP BY sku_code HAVING COUNT(*) > 1
    )
    ORDER BY sku_code, quantity_kg DESC
""")

## Q17

For premix PREMIX-PW-ALOE-009, trace forward to every finished SKU that depends on it (through its parent bulk intermediate). How many SKUs would be affected if this premix had a quality issue?

In [ ]:
# Trace: premix -> used in bulk formula -> bulk intermediate -> used in SKU formula -> SKU
df = run_sql(conn, """
    SELECT DISTINCT s.sku_code, s.name as sku_name, s.is_active,
           bi_parent.bulk_code as parent_bulk,
           bi_premix.bulk_code as premix_code
    FROM bulk_intermediates bi_premix
    JOIN formula_ingredients fi1 ON fi1.ingredient_id = bi_premix.id
    JOIN formulas f1 ON fi1.formula_id = f1.id AND f1.bom_level = 1
    JOIN bulk_intermediates bi_parent ON f1.product_id = bi_parent.id
    JOIN formula_ingredients fi0 ON fi0.ingredient_id = bi_parent.id
    JOIN formulas f0 ON fi0.formula_id = f0.id AND f0.bom_level = 0
    JOIN skus s ON f0.product_id = s.id
    WHERE bi_premix.bulk_code = 'PREMIX-PW-ALOE-009'
    ORDER BY s.sku_code
""")
display(df)
print(f"\nTotal affected SKUs: {len(df)}")

## Q18

What percentage of SKUs have been renamed (have a -OLD alias)? Break down by product category.

In [ ]:
run_sql(conn, """
    SELECT category,
           COUNT(*) as total_skus,
           COUNT(*) FILTER (WHERE sku_code LIKE '%-OLD') as old_alias_count,
           ROUND(100.0 * COUNT(*) FILTER (WHERE sku_code LIKE '%-OLD') / COUNT(*), 1) as pct_renamed
    FROM skus
    GROUP BY category
    ORDER BY pct_renamed DESC
""")

In [ ]:
conn.close()
print("Session closed.")